# Sentiment Analysis Project

This notebook builds two sentiment classifiers (Logistic Regression and LSTM) using Flipkart reviews.

## 1. Setup

In [10]:
# Ensure required packages are available in this notebook kernel
import importlib.util
from IPython import get_ipython
ip = get_ipython()
def ensure_package(package, import_name=None, version=None):
    name = import_name or package
    if importlib.util.find_spec(name) is None and ip is not None:
        pkg_spec = f"{package}=={version}" if version else package
        ip.run_line_magic("pip", f"install {pkg_spec}")
core_packages = [
    ("numpy", "numpy", None),
    ("pandas", "pandas", None),
    ("matplotlib", "matplotlib", None),
    ("scikit-learn", "sklearn", None),
    ("seaborn", "seaborn", None),
    ("joblib", "joblib", None),
    ("nltk", "nltk", None),
    ("tensorflow", "tensorflow", "2.16.1"),
 ]
for pkg, mod, ver in core_packages:
    ensure_package(pkg, mod, ver)

In [12]:
import os
import re
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
import joblib
import nltk
from nltk.corpus import stopwords

TF_AVAILABLE = True
try:
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Embedding, LSTM, Dense
except Exception as e:
    TF_AVAILABLE = False
    print(f"TensorFlow not available in this kernel: {e}")

# Create required folders
os.makedirs('results', exist_ok=True)
os.makedirs('models', exist_ok=True)

# NLTK setup
nltk.download('stopwords', quiet=True)

TensorFlow not available in this kernel: No module named 'tensorflow.python'


True

## 2. Load and Clean Dataset

In [ ]:
df = pd.read_csv('data/flipkart.csv')

# Normalize column names for robust selection
_cols = list(df.columns)
_cols_lower = {c.lower().strip(): c for c in _cols}

# Try common alternatives if exact names differ
review_candidates = ['review', 'reviews', 'review_text', 'text', 'comment', 'comments']
sentiment_candidates = ['sentiment', 'label', 'polarity', 'rating', 'score']

review_col = next(( _cols_lower[c] for c in review_candidates if c in _cols_lower), None)
sentiment_col = next(( _cols_lower[c] for c in sentiment_candidates if c in _cols_lower), None)

if review_col is None or sentiment_col is None:
    raise KeyError(
        f"Expected columns not found. Available columns: {_cols}"
    )

# Keep only required columns
df = df[[review_col, sentiment_col]]
df.columns = ['review', 'sentiment']

# Drop missing values
df = df.dropna()

# Remove neutral sentiments (if present)
df = df[df['sentiment'].astype(str).str.lower() != 'neutral']

# Convert labels (handles common variations)
df['sentiment'] = df['sentiment'].astype(str).str.lower().map({
    'positive': 1,
    'pos': 1,
    'negative': 0,
    'neg': 0,
    '1': 1,
    '0': 0
})

# Drop rows with unmapped labels
df = df.dropna(subset=['sentiment'])
df['sentiment'] = df['sentiment'].astype(int)

# Reduce dataset size
df = df.head(15000).reset_index(drop=True)

df.head()

KeyError: "None of [Index(['review', 'sentiment'], dtype='str')] are in the [columns]"

## 3. Text Preprocessing

In [14]:
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Lowercase
    text = text.lower()
    # Remove special characters
    text = re.sub(r'[^a-z\s]', ' ', text)
    # Tokenize using split()
    tokens = text.split()
    # Remove stopwords
    tokens = [t for t in tokens if t not in stop_words]
    return ' '.join(tokens)

df['clean_review'] = df['review'].astype(str).apply(preprocess_text)

df[['clean_review', 'sentiment']].head()

KeyError: 'review'

## 4. Sentiment Distribution

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='sentiment', data=df)
plt.title('Sentiment Distribution')
plt.xlabel('Sentiment (0=Negative, 1=Positive)')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('results/sentiment_distribution.png')
plt.show()

## 5. Train/Test Split

In [ ]:
X = df['clean_review'].values
y = df['sentiment'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 6A. Feature Engineering + Logistic Regression (TF-IDF)

In [ ]:
tfidf = TfidfVectorizer(max_features=20000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_tfidf, y_train)

y_pred_lr = log_reg.predict(X_test_tfidf)
acc_lr = accuracy_score(y_test, y_pred_lr)
print(f'Logistic Regression Accuracy: {acc_lr:.4f}')
print(classification_report(y_test, y_pred_lr))

In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)
plt.figure(figsize=(5,4))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues')
plt.title('Logistic Regression Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('results/confusion_matrix_logistic.png')
plt.show()

In [ ]:
# Save logistic regression model
joblib.dump({'model': log_reg, 'vectorizer': tfidf}, 'models/logistic_model.pkl')

## 6B. Feature Engineering + LSTM

In [ ]:
if TF_AVAILABLE:
    max_words = 10000
    max_len = 100

    tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
    tokenizer.fit_on_texts(X_train)

    X_train_seq = tokenizer.texts_to_sequences(X_train)
    X_test_seq = tokenizer.texts_to_sequences(X_test)

    X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
    X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')
else:
    print("Skipping LSTM setup because TensorFlow is unavailable.")

In [ ]:
if TF_AVAILABLE:
    lstm_model = Sequential([
        Embedding(input_dim=max_words, output_dim=64, input_length=max_len),
        LSTM(64),
        Dense(1, activation='sigmoid')
    ])

    lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    history = lstm_model.fit(
        X_train_pad, y_train,
        epochs=2,
        batch_size=64,
        validation_split=0.2,
        verbose=1
    )
else:
    lstm_model = None

In [ ]:
if TF_AVAILABLE and lstm_model is not None:
    # LSTM evaluation
    y_pred_lstm_prob = lstm_model.predict(X_test_pad)
    y_pred_lstm = (y_pred_lstm_prob > 0.5).astype(int).ravel()

    acc_lstm = accuracy_score(y_test, y_pred_lstm)
    print(f'LSTM Accuracy: {acc_lstm:.4f}')
    print(classification_report(y_test, y_pred_lstm))
else:
    acc_lstm = None

In [ ]:
if TF_AVAILABLE and lstm_model is not None:
    cm_lstm = confusion_matrix(y_test, y_pred_lstm)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm_lstm, annot=True, fmt='d', cmap='Greens')
    plt.title('LSTM Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig('results/confusion_matrix_lstm.png')
    plt.show()

In [ ]:
if TF_AVAILABLE and lstm_model is not None:
    # Save LSTM model and tokenizer
    lstm_model.save('models/lstm_model.h5')
    with open('models/tokenizer.pkl', 'wb') as f:
        pickle.dump(tokenizer, f)

## 7. Accuracy Comparison

In [ ]:
plt.figure(figsize=(6,4))
models = ['Logistic Regression']
accs = [acc_lr]
if acc_lstm is not None:
    models.append('LSTM')
    accs.append(acc_lstm)
sns.barplot(x=models, y=accs)
plt.ylim(0, 1)
plt.title('Accuracy Comparison')
plt.ylabel('Accuracy')
plt.tight_layout()
plt.savefig('results/accuracy_comparison.png')
plt.show()

## 8. Summary

All models, plots, and artifacts are saved in the `models/` and `results/` folders.